In [1]:
# add this cell to every notebook!

# mount google drive and set project root (required for any data or code loading)
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
PROJECT_ROOT = Path("/content/drive/MyDrive/Team 2/code/collab-sol")
DATA_ROOT = Path("/content/drive/MyDrive/Team 2/data/")
MODEL_RUN_ROOT = Path("/content/drive/MyDrive/Team 2/code/runs/")
%cd {PROJECT_ROOT}

# installs the collab-sol package so we can use the code we develop over the semester
%pip install -q .

Mounted at /content/drive
/content/drive/.shortcut-targets-by-id/1aybJZ4jXd3-7a8OKt1zUUJbhHg1Pb-Uv/Team 2/code/collab-sol
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


# Severity FT-Transformer (GPU)

Runs the two FT-Transformer entries of the severity panel (`scripts/train_severity_models.py`): `classification_ft_transformer` and `regression_ft_transformer`, with the panel's exact settings (50 Optuna trials, 50 max epochs, patience 5, batch size 64, seed 42) on the incident-only cohort. The other six panel models were run on CPU; the FT-Transformer is too slow there (9.3 min per epoch across the four folds).

**Runtime > Change runtime type > T4 GPU** before running.

The team cell above mounts the Drive and sets the data and runs folders; the code itself comes from a pinned GitHub checkout (next cell), because the Drive copy is older than the severity panel.

In [2]:
# The shared Drive copy of collab-sol (Nov 8) predates the severity panel scripts
# (commit 25cfab3, Nov 15). Run from a GitHub checkout pinned to that commit, the
# same code the other six panel models were run with. The Drive copy is not touched.
COLLAB_SOL_COMMIT = "25cfab3"
!rm -rf /content/collab-sol
!git clone -q https://github.com/helloimfrei/collab-sol.git /content/collab-sol
!git -C /content/collab-sol checkout -q {COLLAB_SOL_COMMIT}
!git -C /content/collab-sol log -1 --format="collab-sol commit %h %ad %s"
%cd /content/collab-sol
%pip install -q ".[models]"

import torch
assert torch.cuda.is_available(), "No GPU: set Runtime > Change runtime type > T4 GPU"
print(torch.__version__, torch.cuda.get_device_name(0))


collab-sol commit 25cfab3 Sun Nov 15 16:28:37 2026 -0500 Add severity classification and regression panel
/content/collab-sol
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
2.14.1+cu130 Tesla T4


In [4]:
# Incident-only file (10,206 rows, 3 MB). If it isn't on the Drive, the full
# final_dataset.parquet works too: the scripts keep only incident == 1 rows.
candidates = [DATA_ROOT / "processed" / "final_dataset.parquet",
              DATA_ROOT / "processed" / "final_dataset.parquet"]
DATASET = next((p for p in candidates if p.is_file()), None)
assert DATASET is not None, f"Dataset not found; looked for {[str(p) for p in candidates]}"
print("Dataset:", DATASET)

# One folder for both runs, panel layout. Keep the same name if the two runs happen in separate sessions.
RUN_GROUP = MODEL_RUN_ROOT / "severity_ft_transformer_2026-11-16"
print("Outputs:", RUN_GROUP)

Dataset: /content/drive/MyDrive/Team 2/data/processed/final_dataset.parquet
Outputs: /content/drive/MyDrive/Team 2/code/runs/severity_ft_transformer_2026-11-16


## 1. Timing check (a minute or so)
One trial, one epoch, written to `/content` (not the Drive). On the local CPU this took **9.3 min**. Rough full-run time per model: (this cell's minutes) × 50 trials × ~20 epochs, more for the bigger configs the search tries.

In [5]:
!python scripts/train_severity_classification.py --model ft_transformer   --dataset "{DATASET}" --output-dir /content/timing_check   --trials 1 --max-epochs 1 --patience 1 --batch-size 64 --seed 42 --require-cuda

severity_target  property_damage_only  injury  fatal
asof_year                                           
2019                             1041     465    257
2020                              877     424    182
2021                             1045     413    219
2022                             1070     463    249
2023                             1040     468    237
2024                             1043     467    246
Dataset: /content/drive/.shortcut-targets-by-id/1aybJZ4jXd3-7a8OKt1zUUJbhHg1Pb-Uv/Team 2/data/processed/final_dataset.parquet
Using 224 predictors: 187 numeric + 37 categorical
[I 2026-11-16 01:27:44,127] A new study created in memory with name: conditional_severity_classification_ft_transformer
trial 1/1 | fold 1/4 | preparing data
epoch 1/1 | loss 0.95496 | validation log loss 0.9260 (best 0.9260 @ 1) | 6.6 s/epoch
trial 1/1 | fold 1/4 | log loss 0.9260 | 13.5 s
trial 1/1 | fold 2/4 | preparing data
epoch 1/1 | loss 0.95415 | validation log loss 0.9070 (best 0.9070 @ 

## 2. Full runs
Same command line `train_severity_models.py` builds for these two models. A run cut off midway can't resume: delete its half-written folder under `RUN_GROUP` and rerun the cell (the scripts refuse a non-empty output folder). If the timing check says both won't fit in one session, run them in separate sessions.

In [6]:
!python scripts/train_severity_classification.py --model ft_transformer   --dataset "{DATASET}" --output-dir "{RUN_GROUP}/classification_ft_transformer"   --trials 50 --max-epochs 50 --patience 5 --n-jobs -1 --xgboost-device cpu --seed 42   --batch-size 64 --require-cuda

severity_target  property_damage_only  injury  fatal
asof_year                                           
2019                             1041     465    257
2020                              877     424    182
2021                             1045     413    219
2022                             1070     463    249
2023                             1040     468    237
2024                             1043     467    246
Dataset: /content/drive/.shortcut-targets-by-id/1aybJZ4jXd3-7a8OKt1zUUJbhHg1Pb-Uv/Team 2/data/processed/final_dataset.parquet
Using 224 predictors: 187 numeric + 37 categorical
[I 2026-11-16 01:28:31,585] A new study created in memory with name: conditional_severity_classification_ft_transformer
trial 1/50 | fold 1/4 | preparing data
epoch 1/50 | loss 0.95496 | validation log loss 0.9260 (best 0.9260 @ 1) | 3.8 s/epoch
epoch 2/50 | loss 0.94719 | validation log loss 0.9241 (best 0.9241 @ 2) | 1.6 s/epoch
epoch 3/50 | loss 0.94569 | validation log loss 0.9310 (best 0.924

In [7]:
!python scripts/train_severity_regression.py --model ft_transformer   --dataset "{DATASET}" --output-dir "{RUN_GROUP}/regression_ft_transformer"   --trials 50 --max-epochs 50 --patience 5 --n-jobs -1 --xgboost-device cpu --seed 42   --batch-size 64 --require-cuda

           crash_rows  mean_severity
asof_year                           
2019             1763       1.555303
2020             1483       1.531355
2021             1677       1.507454
2022             1782       1.539282
2023             1745       1.539828
2024             1756       1.546128
Dataset: /content/drive/.shortcut-targets-by-id/1aybJZ4jXd3-7a8OKt1zUUJbhHg1Pb-Uv/Team 2/data/processed/final_dataset.parquet
Using 224 predictors: 187 numeric + 37 categorical
[I 2026-11-16 03:58:22,333] A new study created in memory with name: conditional_severity_regression_ft_transformer
trial 1/50 | fold 1/4 | preparing data
epoch 1/50 | loss 0.57481 | validation MAE 0.6381 (best 0.6381 @ 1) | 4.3 s/epoch
epoch 2/50 | loss 0.53919 | validation MAE 0.6191 (best 0.6191 @ 2) | 1.7 s/epoch
epoch 3/50 | loss 0.53915 | validation MAE 0.6300 (best 0.6191 @ 2) | 1.7 s/epoch
epoch 4/50 | loss 0.52716 | validation MAE 0.6114 (best 0.6114 @ 4) | 1.8 s/epoch
epoch 5/50 | loss 0.51557 | validation MAE 0